# 28 - Scale-safe conv at 30M: does the gate keep the gain?

At 3M the zero-init gate removed conv's +20% harm (gated -0.7% vs combo). The decisive question:
at ~30M, where plain conv WINS by -10.2%, does the gated conv keep that gain?

* combo (30M)            = 55.46  (reference, from notebook 22/23)
* combo_conv3 (30M)      = 49.78  (-10.2%, the gain)
* combo_conv3_gated (30M)= ??     <- this run

If gated ~= 49.8 -> IDEAL: a depthwise conv that is SAFE to add at any scale (no harm small, full gain
large). If gated ~= 55 -> gate too conservative (safe but no gain). 2 seeds, checkpointed every 500.


In [1]:
%matplotlib inline
import os, math, time, csv, json, gc
from contextlib import nullcontext
import numpy as np
import torch
from model import Config, GPT, count_params

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
D = "../dataset"
train_data = torch.from_numpy(np.load(D + "/wt103_bpe_train.npy"))
val_data = torch.from_numpy(np.load(D + "/wt103_bpe_val.npy"))
vocab = json.load(open(D + "/wt103_bpe_meta.json"))["vocab"]
print(f"vocab={vocab}  train={len(train_data):,}  val={len(val_data):,}")

BLOCK, BATCH = 256, 16
CKPT_DIR = "../results/scale_ckpt_safe"
os.makedirs(CKPT_DIR, exist_ok=True)

def get_batch(split, gen):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK, (BATCH,), generator=gen)
    x = torch.stack([d[i:i + BLOCK] for i in ix]); y = torch.stack([d[i + 1:i + 1 + BLOCK] for i in ix])
    return x.to(device), y.to(device)
def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

def train_one(name, over, seed=0, max_iters=6000, eval_iters=200, lr=3e-4, warmup=200, min_lr=3e-5, ckpt_every=500):
    ckpt_path = f"{CKPT_DIR}/{name}_s{seed}.pt"
    torch.manual_seed(seed)
    cfg = Config(vocab_size=vocab, block_size=BLOCK, n_layer=8, n_head=8, n_embd=512, dropout=0.1, **over)
    model = GPT(cfg).to(device)
    pm = count_params(model) / 1e6
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.1)
    tg = torch.Generator().manual_seed(7777 + seed)
    start_it = 0
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); tg.set_state(ck["tg"]); start_it = ck["iter"]
        print(f"    [{name} s{seed}] RESUMED from {start_it}", flush=True)
    def get_lr(it):
        if it < warmup: return lr * (it + 1) / warmup
        r = (it - warmup) / max(1, max_iters - warmup)
        return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * r))
    def save_ckpt(it):
        tmp = ckpt_path + ".tmp"
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "iter": it, "tg": tg.get_state()}, tmp)
        os.replace(tmp, ckpt_path)
    @torch.no_grad()
    def ev():
        model.eval(); g = torch.Generator().manual_seed(4242); tot = 0.0
        for _ in range(eval_iters):
            x, y = get_batch("val", g)
            with autocast_ctx():
                _, loss = model(x, y)
            tot += loss.item()
        model.train(); return tot / eval_iters
    t0 = time.time()
    for it in range(start_it, max_iters):
        for grp in opt.param_groups: grp["lr"] = get_lr(it)
        x, y = get_batch("train", tg)
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if it % 500 == 0 or it == max_iters - 1:
            print(f"    [{name} s{seed}] it {it}/{max_iters} loss {loss.item():.3f} {(time.time()-t0)/60:.1f}m", flush=True)
        if (it + 1) % ckpt_every == 0: save_ckpt(it + 1)
    ppl = math.exp(ev())
    if os.path.exists(ckpt_path): os.remove(ckpt_path)
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return ppl, pm
print("ready")


device: cuda
vocab=8192  train=14,923,628  val=281,169
ready


In [2]:
COMBO = {"pos_encoding": "rope", "activation": "swiglu", "norm_placement": "pre", "norm_type": "rmsnorm"}
OVER = {**COMBO, "conv_kernel": 3, "conv_gate": True}
SEEDS = [0, 1]
runs_csv = "../results/scalesafe30_runs.csv"
done = set()
if os.path.exists(runs_csv):
    with open(runs_csv) as f:
        r = csv.reader(f); next(r, None)
        for row in r:
            if len(row) == 3: done.add(row[0])
else:
    with open(runs_csv, "w", newline="") as f:
        csv.writer(f).writerow(["seed", "params_M", "val_ppl"])
t0 = time.time()
for s in SEEDS:
    if str(s) in done:
        print(f"seed {s} done, skip", flush=True); continue
    p, pm = train_one("combo_conv3_gated", OVER, seed=s, max_iters=6000)
    with open(runs_csv, "a", newline="") as f:
        csv.writer(f).writerow([s, f"{pm:.3f}", f"{p:.4f}"])
    print(f"seed {s}: ppl {p:.3f} ({pm:.1f}M) | total {(time.time()-t0)/60:.1f} min\n", flush=True)
print(f"ALL DONE in {(time.time()-t0)/60:.1f} min")


    [combo_conv3_gated s0] it 0/6000 loss 9.106 0.0m


    [combo_conv3_gated s0] it 500/6000 loss 5.847 0.9m


    [combo_conv3_gated s0] it 1000/6000 loss 5.254 1.7m


    [combo_conv3_gated s0] it 1500/6000 loss 4.972 2.6m


    [combo_conv3_gated s0] it 2000/6000 loss 4.575 3.4m


    [combo_conv3_gated s0] it 2500/6000 loss 4.420 4.3m


... (19 outputs left out)

    [combo_conv3_gated s1] it 5500/6000 loss 4.014 9.4m


    [combo_conv3_gated s1] it 5999/6000 loss 3.946 10.3m


seed 1: ppl 54.517 (29.4M) | total 20.9 min



ALL DONE in 20.9 min


In [3]:
import numpy as np, csv
def load(fn):
    d = {}
    with open(fn) as f:
        r = csv.reader(f); next(r, None)
        for row in r: d[row[0]] = float(row[-1])
    return d
gated_vals = []
with open('../results/scalesafe30_runs.csv') as f:
    r = csv.reader(f); next(r, None)
    for row in r: gated_vals.append(float(row[-1]))
gated = float(np.mean(gated_vals))
s0 = load('../results/scale_confirm_runs.csv'); s1 = load('../results/scale_confirm_s1_runs.csv')
combo = np.mean([s0['combo'], s1['combo']]); conv = np.mean([s0['combo_conv3'], s1['combo_conv3']])
print(f"30M, WT103 BPE:")
print(f"  combo               {combo:.3f}")
print(f"  combo_conv3 (plain) {conv:.3f}  ({100*(conv-combo)/combo:+.2f}% vs combo)")
print(f"  combo_conv3_gated   {gated:.3f}  ({100*(gated-combo)/combo:+.2f}% vs combo)")
gain_kept = 100*(gated-combo)/combo
print(f"\ngated keeps {gain_kept:+.1f}% vs combo (plain conv was {100*(conv-combo)/combo:+.1f}%)")
if gain_kept < -6:
    print("=> IDEAL: gated conv keeps most/all of the gain AND was harmless at 3M -> safe at any scale.")
elif gain_kept < -2:
    print("=> PARTIAL WIN: keeps some gain; safe but leaves some on the table.")
else:
    print("=> gate too conservative: safe but suppresses the 30M gain.")


30M, WT103 BPE:
  combo               55.455
  combo_conv3 (plain) 49.780  (-10.23% vs combo)
  combo_conv3_gated   54.833  (-1.12% vs combo)

gated keeps -1.1% vs combo (plain conv was -10.2%)
=> gate too conservative: safe but suppresses the 30M gain.


## Reading this

Combined with notebook 27 (3M: gate removed the +20% harm), this answers whether the zero-init gated
conv is a genuinely useful module: harmless at small scale AND beneficial at large scale = safe to add
at any scale. That combination (not the gate itself, which is ReZero) is the candidate contribution.
